### Импорты и подготовка данных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# воспроизводимость
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем: {device}")

#### Нормализация

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].values)

#### Нарезка на скользящие окна

In [ ]:
LSTM работает с последовательностями, поэтому режем ряд на окна.

def create_sequences(data, seq_len, step=1):
    """Превращает [N, features] в [num_windows, seq_len, features]"""
    sequences = []
    indices = []  # запоминаем конец окна (для привязки ко времени)
    for i in range(0, len(data) - seq_len + 1, step):
        sequences.append(data[i:i + seq_len])
        indices.append(i + seq_len - 1)
    return np.array(sequences), np.array(indices)

SEQ_LEN = 30   # длина окна (подберите под частоту ваших данных)
X_seq, seq_idx = create_sequences(X_scaled, SEQ_LEN, step=1)
print("Форма окон:", X_seq.shape)  # [num_windows, SEQ_LEN, n_features]

#### Модель LSTM Autoencoder

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, embedding_dim=32, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features

        # Encoder: сжимает последовательность в вектор
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=embedding_dim,
            num_layers=1,
            batch_first=True
        )
        # Decoder: восстанавливает последовательность
        self.decoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=embedding_dim,
            num_layers=1,
            batch_first=True
        )
        self.output_layer = nn.Linear(embedding_dim, n_features)

    def forward(self, x):
        # Кодирование
        _, (hidden, _) = self.encoder(x)         # hidden: [1, batch, emb_dim]
        # Повторяем скрытый вектор на всю длину последовательности
        latent = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        # Декодирование
        decoded, _ = self.decoder(latent)
        out = self.output_layer(decoded)
        return out

In [ ]:
# DataLoader
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
dataset = TensorDataset(X_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = LSTMAutoencoder(
    n_features=len(features),
    embedding_dim=32,
    seq_len=SEQ_LEN
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20
history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss: {avg_loss:.5f}")

# График обучения
plt.figure(figsize=(8,3))
plt.plot(history)
plt.title('Loss при обучении')
plt.xlabel('Эпоха'); plt.ylabel('MSE')
plt.show()

#### Подсчёт ошибки реконструкции и поиск аномалий

In [ ]:
model.eval()
errors = []

with torch.no_grad():
    for i in range(0, len(X_tensor), 256):
        batch = X_tensor[i:i+256].to(device)
        recon = model(batch)
        # ошибка для каждого окна (усредняем по времени и признакам)
        err = torch.mean((recon - batch)**2, dim=(1, 2))
        errors.extend(err.cpu().numpy())

errors = np.array(errors)

# Порог: например, 99-й перцентиль ошибки
threshold = np.percentile(errors, 99)
print(f"Порог: {threshold:.4f}")

anomaly_windows = errors > threshold

# Привязываем аномалии обратно к строкам датафрейма
df['recon_error'] = np.nan
df.loc[seq_idx, 'recon_error'] = errors
df['anomaly'] = False
df.loc[seq_idx[anomaly_windows], 'anomaly'] = True

print(f"Найдено аномальных точек: {df['anomaly'].sum()}")

#### Визуализация — ошибка реконструкции

In [ ]:
plt.figure(figsize=(15, 4))
plt.plot(seq_idx, errors, label='Ошибка реконструкции', color='steelblue')
plt.axhline(threshold, color='red', linestyle='--', label='Порог')
plt.scatter(seq_idx[anomaly_windows], errors[anomaly_windows],
            color='red', s=20, label='Аномалии')
plt.title('Ошибка реконструкции по времени')
plt.xlabel('Индекс времени'); plt.ylabel('MSE')
plt.legend()
plt.show()

#### Визуализация — аномалии на признаках

In [ ]:
# Покажем 4 главных признака с подсветкой аномалий
plot_features = ['высота', 'скорость', 'тангаж', 'крен']

fig, axes = plt.subplots(len(plot_features), 1, figsize=(15, 10), sharex=True)

for ax, col in zip(axes, plot_features):
    ax.plot(df.index, df[col], color='steelblue', lw=0.8)
    anom = df[df['anomaly']]
    ax.scatter(anom.index, anom[col], color='red', s=15, zorder=5)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)

axes[0].set_title('Аномалии на признаках (красные точки)')
axes[-1].set_xlabel('Время')
plt.tight_layout()
plt.show()

#### Визуализация — PCA (общая картина)

In [ ]:
# Берём по одной точке на окно (последняя точка окна)
X_points = X_scaled[seq_idx]
pca = PCA(n_components=2)
emb = pca.fit_transform(X_points)

plt.figure(figsize=(8, 6))
plt.scatter(emb[~anomaly_windows, 0], emb[~anomaly_windows, 1],
            c='steelblue', s=10, label='Норма', alpha=0.5)
plt.scatter(emb[anomaly_windows, 0], emb[anomaly_windows, 1],
            c='red', s=30, label='Аномалия')
plt.title('PCA проекция (2D)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend()
plt.show()

#### Heatmap — какие признаки "виноваты"

In [ ]:
Покажет, по каким именно признакам была наибольшая ошибка.

# Считаем ошибку по каждому признаку отдельно для аномальных окон
model.eval()
with torch.no_grad():
    anom_idx = np.where(anomaly_windows)[0]
    if len(anom_idx) > 0:
        sample = X_tensor[anom_idx].to(device)
        recon = model(sample)
        # ошибка по признакам: усредняем по времени
        feat_error = torch.mean((recon - sample)**2, dim=1).cpu().numpy()

        plt.figure(figsize=(12, 6))
        sns.heatmap(feat_error.T, cmap='Reds',
                    yticklabels=features,
                    cbar_kws={'label': 'Ошибка'})
        plt.title('Вклад каждого признака в аномалии')
        plt.xlabel('Номер аномального окна')
        plt.ylabel('Признак')
        plt.tight_layout()
        plt.show()

In [ ]:
Важные замечания


Параметр	Что делать
SEQ_LEN	Подберите под частоту данных. Если запись 1 Гц — окно 30 = 30 сек
threshold (перцентиль)	99% — старт. Меньше процент → больше аномалий
embedding_dim	Слишком большой → модель запомнит всё, включая аномалии
Разные полёты	Не смешивайте! Режьте окна внутри одного полёта


In [ ]:
# производные — как быстро меняются признаки
for col in ['высота', 'скорость', 'тангаж', 'крен']:
    df[f'{col}_diff'] = df.groupby('aircraft_id')[col].diff()

# заполняем NaN (первая строка каждого аппарата)
df = df.fillna(0)

# добавляем новые признаки в список
features = features + ['высота_diff', 'скорость_diff', 'тангаж_diff', 'крен_diff']

In [ ]:
# 5. Нарезка окон ПО КАЖДОМУ АППАРАТУ
def create_sequences_grouped(df, features, seq_len, id_column, step=1):
    all_seq, all_idx, all_ids = [], [], []
    for aid, group in df.groupby(id_column):
        group = group.sort_index()
        data = group[features].values
        gidx = group.index.values
        for i in range(0, len(data) - seq_len + 1, step):
            all_seq.append(data[i:i + seq_len])
            all_idx.append(gidx[i + seq_len - 1])
            all_ids.append(aid)
    return np.array(all_seq), np.array(all_idx), np.array(all_ids)

SEQ_LEN = 30
X_seq, seq_idx, seq_ids = create_sequences_grouped(
    df, features, SEQ_LEN, id_column='aircraft_id'
)
print("Окон:", X_seq.shape)

In [ ]:
hj

In [ ]:
all_sequences = []
all_indices = []

for machine_id, group in df.groupby("machine_id"):
    X_machine = group[features].values

    seqs, idx = create_sequences(X_machine, SEQ_LEN)

    # индексы относительно исходного df
    global_idx = group.index[idx]

    all_sequences.append(seqs)
    all_indices.append(global_idx)

X_seq = np.concatenate(all_sequences)
seq_idx = np.concatenate(all_indices)

In [ ]:
# 5. Нарезка окон ПО КАЖДОМУ АППАРАТУ
def create_sequences_grouped(df, features, seq_len, id_column, step=1):
    all_seq, all_idx, all_ids = [], [], []
    for aid, group in df.groupby(id_column):
        group = group.sort_index()
        data = group[features].values
        gidx = group.index.values
        for i in range(0, len(data) - seq_len + 1, step):
            all_seq.append(data[i:i + seq_len])
            all_idx.append(gidx[i + seq_len - 1])
            all_ids.append(aid)
    return np.array(all_seq), np.array(all_idx), np.array(all_ids)

SEQ_LEN = 30
X_seq, seq_idx, seq_ids = create_sequences_grouped(
    df, features, SEQ_LEN, id_column='aircraft_id'
)
print("Окон:", X_seq.shape)

In [ ]:
# Чтобы модель ловила именно резкие изменения (а не стиль вождения), добавь скорость изменения признаков:

# производные — как быстро меняются признаки
for col in ['высота', 'скорость', 'тангаж', 'крен']:
    df[f'{col}_diff'] = df.groupby('aircraft_id')[col].diff()

# заполняем NaN (первая строка каждого аппарата)
df = df.fillna(0)

# добавляем новые признаки в список
features = features + ['высота_diff', 'скорость_diff', 'тангаж_diff', 'крен_diff']

In [ ]:
# ============================================================
#  ДЕТЕКЦИЯ АНОМАЛИЙ ЛЕТАТЕЛЬНЫХ АППАРАТОВ (LSTM Autoencoder)
#  Unsupervised, разделение train/test ПО АППАРАТАМ
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# воспроизводимость
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")


# ============================================================
#  1. ЗАГРУЗКА ДАННЫХ
# ============================================================
# df = pd.read_csv('your_data.csv')



# сортируем: важно, чтобы данные шли по порядку внутри каждого аппарата
df = df.sort_values([ID_COLUMN, TIME_COLUMN]).reset_index(drop=True)

print(f"Всего строк: {len(df)}")
print(f"Всего аппаратов: {df[ID_COLUMN].nunique()}")


# ============================================================
#  2. ДОБАВЛЯЕМ ПРИЗНАКИ-ПРОИЗВОДНЫЕ (как быстро меняются значения)
#     Помогает ловить РЕЗКИЕ сбои, а не стиль вождения оператора
# ============================================================
diff_features = ['высота', 'скорость', 'тангаж', 'крен']
for col in diff_features:
    df[f'{col}_diff'] = df.groupby(ID_COLUMN)[col].diff()

df = df.fillna(0)  # первая строка каждого аппарата → NaN после diff
features = features + [f'{col}_diff' for col in diff_features]

print(f"Итого признаков для модели: {len(features)}")


# ============================================================
#  3. РАЗДЕЛЕНИЕ TRAIN / TEST ПО АППАРАТАМ
#     (не по строкам! целые аппараты идут либо в train, либо в test)
# ============================================================
all_aircraft = df[ID_COLUMN].unique()
np.random.shuffle(all_aircraft)

split = int(len(all_aircraft) * 0.8)
train_aircraft = all_aircraft[:split]
test_aircraft  = all_aircraft[split:]

print(f"Аппаратов в train: {len(train_aircraft)}")
print(f"Аппаратов в test : {len(test_aircraft)}")

df_train = df[df[ID_COLUMN].isin(train_aircraft)].copy()
df_test  = df[df[ID_COLUMN].isin(test_aircraft)].copy()


# ============================================================
#  4. НОРМАЛИЗАЦИЯ
#     scaler обучаем ТОЛЬКО на train, применяем к обоим
#     (аппараты одинаковые → общий scaler корректен)
# ============================================================
scaler = StandardScaler()
df_train[features] = scaler.fit_transform(df_train[features])
df_test[features]  = scaler.transform(df_test[features])


# ============================================================
#  5. НАРЕЗКА НА ОКНА (отдельно для каждого аппарата)
#     Окна НЕ пересекают границы между аппаратами
# ============================================================
def create_sequences_grouped(df, features, seq_len, id_column, step=1):
    all_seq, all_idx, all_ids = [], [], []
    for aid, group in df.groupby(id_column):
        group = group.sort_index()
        data = group[features].values
        gidx = group.index.values          # реальные индексы строк в df
        for i in range(0, len(data) - seq_len + 1, step):
            all_seq.append(data[i:i + seq_len])
            all_idx.append(gidx[i + seq_len - 1])  # привязка к концу окна
            all_ids.append(aid)
    return (np.array(all_seq),
            np.array(all_idx),
            np.array(all_ids))

SEQ_LEN = 30   # длина окна — подбери под частоту данных

X_train_seq, train_idx, train_ids = create_sequences_grouped(
    df_train, features, SEQ_LEN, ID_COLUMN
)
X_test_seq, test_idx, test_ids = create_sequences_grouped(
    df_test, features, SEQ_LEN, ID_COLUMN
)

print(f"Окон train: {X_train_seq.shape}")
print(f"Окон test : {X_test_seq.shape}")


# ============================================================
#  6. МОДЕЛЬ: LSTM AUTOENCODER
# ============================================================
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, embedding_dim=32, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(n_features, embedding_dim,
                               batch_first=True)
        self.decoder = nn.LSTM(embedding_dim, embedding_dim,
                               batch_first=True)
        self.output_layer = nn.Linear(embedding_dim, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        latent = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(latent)
        return self.output_layer(decoded)


model = LSTMAutoencoder(
    n_features=len(features),
    embedding_dim=32,
    seq_len=SEQ_LEN
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# ============================================================
#  7. ОБУЧЕНИЕ (train) + ВАЛИДАЦИЯ (test-аппараты)
# ============================================================
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t),  batch_size=64, shuffle=False)

EPOCHS = 20
history_train, history_val = [], []

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    train_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # --- validation (на тест-аппаратах) ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)
            val_loss += criterion(model(batch), batch).item()

    train_loss /= len(train_loader)
    val_loss /= len(test_loader)
    history_train.append(train_loss)
    history_val.append(val_loss)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"train: {train_loss:.5f} | val: {val_loss:.5f}")

# График обучения
plt.figure(figsize=(8, 3))
plt.plot(history_train, label='train')
plt.plot(history_val, label='val (test-аппараты)')
plt.title('Loss при обучении')
plt.xlabel('Эпоха'); plt.ylabel('MSE')
plt.legend(); plt.grid(alpha=0.3)
plt.show()


# ============================================================
#  8. ПОДСЧЁТ ОШИБКИ РЕКОНСТРУКЦИИ
#     Функция считает ошибку для любого набора окон
# ============================================================
def compute_errors(model, X_seq, batch=256):
    model.eval()
    X_t = torch.tensor(X_seq, dtype=torch.float32)
    errors = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch):
            b = X_t[i:i+batch].to(device)
            recon = model(b)
            err = torch.mean((recon - b)**2, dim=(1, 2))
            errors.extend(err.cpu().numpy())
    return np.array(errors)

# Считаем ошибку на ТЕСТ-аппаратах (которых модель не видела)
test_errors = compute_errors(model, X_test_seq)


# ============================================================
#  9. ПОИСК АНОМАЛИЙ (порог по каждому аппарату отдельно)
# ============================================================
# Записываем результаты обратно в df_test
df_test['recon_error'] = np.nan
df_test.loc[test_idx, 'recon_error'] = test_errors
df_test['anomaly'] = False
df_test['seq_id'] = np.nan
df_test.loc[test_idx, 'seq_id'] = test_ids

PERCENTILE = 99  # порог: верхний 1% ошибок = аномалии

# Порог СВОЙ для каждого аппарата
for aid in test_aircraft:
    mask = test_ids == aid
    if mask.sum() == 0:
        continue
    aircraft_errors = test_errors[mask]
    threshold = np.percentile(aircraft_errors, PERCENTILE)

    # индексы окон-аномалий этого аппарата
    anom_mask = aircraft_errors > threshold
    anom_global_idx = test_idx[mask][anom_mask]
    df_test.loc[anom_global_idx, 'anomaly'] = True

print(f"Всего аномалий найдено: {df_test['anomaly'].sum()}")

# Статистика по аппаратам
print("\nАномалий по аппаратам:")
stats = df_test[df_test['anomaly']].groupby(ID_COLUMN).size()
print(stats.sort_values(ascending=False))


# ============================================================
#  10. ВИЗУАЛИЗАЦИЯ ДЛЯ ОДНОГО АППАРАТА
#      (поменяй TARGET_ID на нужный)
# ============================================================
TARGET_ID = test_aircraft[0]   # <<< сюда впиши нужный id аппарата

sub = df_test[df_test[ID_COLUMN] == TARGET_ID].copy()
anom = sub[sub['anomaly']]

# --- 10.1 Признаки с подсветкой аномалий ---
plot_features = ['высота', 'скорость', 'тангаж', 'крен']
fig, axes = plt.subplots(len(plot_features), 1, figsize=(15, 10), sharex=True)

for ax, col in zip(axes, plot_features):
    ax.plot(sub.index, sub[col], color='steelblue', lw=0.8, label=col)
    ax.scatter(anom.index, anom[col], color='red', s=20, zorder=5,
               label='аномалия')
    ax.set_ylabel(col)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_title(f'Аппарат #{TARGET_ID} — признаки и аномалии')
axes[-1].set_xlabel('Время (индекс)')
plt.tight_layout()
plt.show()

# --- 10.2 Ошибка реконструкции этого аппарата ---
plt.figure(figsize=(15, 4))
plt.plot(sub.index, sub['recon_error'], color='steelblue',
         label='ошибка реконструкции')
threshold = np.percentile(sub['recon_error'].dropna(), PERCENTILE)
plt.axhline(threshold, color='red', linestyle='--', label='порог')
plt.scatter(anom.index, anom['recon_error'], color='red', s=20,
            zorder=5, label='аномалия')
plt.title(f'Аппарат #{TARGET_ID} — ошибка реконструкции')
plt.xlabel('Время (индекс)'); plt.ylabel('MSE')
plt.legend(); plt.grid(alpha=0.3)
plt.show()


# ============================================================
#  11. (опц.) PCA — общая картина по тест-аппаратам
# ============================================================
X_points = X_test_seq[:, -1, :]  # последняя точка каждого окна
emb = PCA(n_components=2).fit_transform(X_points)

is_anom = np.isin(test_idx, df_test[df_test['anomaly']].index)

plt.figure(figsize=(8, 6))
plt.scatter(emb[~is_anom, 0], emb[~is_anom, 1], c='steelblue',
            s=8, alpha=0.4, label='норма')
plt.scatter(emb[is_anom, 0], emb[is_anom, 1], c='red',
            s=25, label='аномалия')
plt.title('PCA проекция (тест-аппараты)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(); plt.show()

In [ ]:
# ============================================================
#  HEATMAP: вклад каждого признака в аномалии (для 1 аппарата)
# ============================================================
TARGET_ID = test_aircraft[0]   # <<< нужный id

# находим окна-аномалии этого аппарата
mask_aircraft = test_ids == TARGET_ID
sub_errors = test_errors[mask_aircraft]
sub_seq = X_test_seq[mask_aircraft]
threshold = np.percentile(sub_errors, PERCENTILE)
anom_local = sub_errors > threshold   # какие окна аппарата аномальны

model.eval()
with torch.no_grad():
    anom_windows = sub_seq[anom_local]
    if len(anom_windows) > 0:
        sample = torch.tensor(anom_windows, dtype=torch.float32).to(device)
        recon = model(sample)
        # ошибка по каждому признаку (усредняем по времени внутри окна)
        feat_error = torch.mean((recon - sample)**2, dim=1).cpu().numpy()
        # feat_error: [кол-во аномальных окон, кол-во признаков]

        plt.figure(figsize=(12, 8))
        sns.heatmap(feat_error.T, cmap='Reds',
                    yticklabels=features,
                    cbar_kws={'label': 'Ошибка'})
        plt.title(f'Аппарат #{TARGET_ID}: вклад признаков в аномалии')
        plt.xlabel('Номер аномального окна')
        plt.ylabel('Признак')
        plt.tight_layout()
        plt.show()
    else:
        print(f"У аппарата {TARGET_ID} нет аномалий")

In [ ]:
# Считаем ошибку по признакам для ВСЕХ окон аппарата (не только аномальных)
model.eval()
with torch.no_grad():
    all_windows = torch.tensor(sub_seq, dtype=torch.float32).to(device)
    recon = model(all_windows)
    feat_err_all = torch.mean((recon - all_windows)**2, dim=1).cpu().numpy()
    # feat_err_all: [кол-во окон аппарата, кол-во признаков]

# Рисуем отдельный график для каждого признака
n = len(features)
fig, axes = plt.subplots(n, 1, figsize=(15, 2.5*n), sharex=True)

for i, (ax, col) in enumerate(zip(axes, features)):
    ax.plot(feat_err_all[:, i], color='steelblue', lw=0.8)
    # подсветим аномальные окна
    anom_points = np.where(anom_local)[0]
    ax.scatter(anom_points, feat_err_all[anom_points, i],
               color='red', s=15, zorder=5)
    ax.set_ylabel(col, fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_title(f'Аппарат #{TARGET_ID}: ошибка по каждому признаку во времени')
axes[-1].set_xlabel('Номер окна')
plt.tight_layout()
plt.show()

In [ ]:
# Применяем обученную модель КО ВСЕМ данным
# (объединяем train + test обратно)

df_all = df.copy()  # все 50 аппаратов
df_all[features] = scaler.transform(df_all[features])  # тот же scaler!

X_all_seq, all_idx, all_ids = create_sequences_grouped(
    df_all, features, SEQ_LEN, ID_COLUMN
)

# считаем ошибку для всех
all_errors = compute_errors(model, X_all_seq)

# заполняем аномалии (порог по каждому аппарату)
df_all['recon_error'] = np.nan
df_all.loc[all_idx, 'recon_error'] = all_errors
df_all['anomaly'] = False

for aid in df_all[ID_COLUMN].unique():
    mask = all_ids == aid
    threshold = np.percentile(all_errors[mask], PERCENTILE)
    anom_idx = all_idx[mask][all_errors[mask] > threshold]
    df_all.loc[anom_idx, 'anomaly'] = True

print(f"Всего аномалий во всех данных: {df_all['anomaly'].sum()}")

In [ ]:
def analyze_aircraft(df_all, all_errors, all_idx, all_ids,
                     aircraft_id, features, percentile=99, show_plots=True):
    """
    Полный анализ аномалий для ОДНОГО аппарата.
    Возвращает информацию и рисует графики.
    """
    print("=" * 60)
    print(f"  АППАРАТ #{aircraft_id}")
    print("=" * 60)

    # данные этого аппарата
    sub = df_all[df_all[ID_COLUMN] == aircraft_id].copy()
    mask = all_ids == aircraft_id
    aircraft_errors = all_errors[mask]
    aircraft_idx = all_idx[mask]

    if len(aircraft_errors) == 0:
        print("Нет данных для этого аппарата")
        return None

    # порог для ЭТОГО аппарата
    threshold = np.percentile(aircraft_errors, percentile)
    anom_mask = aircraft_errors > threshold
    anom_idx = aircraft_idx[anom_mask]

    n_total = len(aircraft_errors)
    n_anom = anom_mask.sum()

    # --- текстовый отчёт ---
    print(f"Всего окон:        {n_total}")
    print(f"Аномальных окон:   {n_anom} ({100*n_anom/n_total:.1f}%)")
    print(f"Порог ошибки:      {threshold:.4f}")
    print(f"Макс. ошибка:      {aircraft_errors.max():.4f}")
    print(f"Средняя ошибка:    {aircraft_errors.mean():.4f}")

    if not show_plots or n_anom == 0:
        if n_anom == 0:
            print("Аномалий не обнаружено.")
        return {'id': aircraft_id, 'n_anomalies': n_anom,
                'anom_idx': anom_idx, 'threshold': threshold}

    anom_rows = df_all.loc[anom_idx]

    # --- ГРАФИК 1: признаки с подсветкой аномалий ---
    plot_features = ['высота', 'скорость', 'тангаж', 'крен']
    fig, axes = plt.subplots(len(plot_features), 1,
                             figsize=(15, 10), sharex=True)
    for ax, col in zip(axes, plot_features):
        ax.plot(sub.index, sub[col], color='steelblue', lw=0.8)
        ax.scatter(anom_rows.index, anom_rows[col],
                   color='red', s=20, zorder=5)
        ax.set_ylabel(col)
        ax.grid(alpha=0.3)
    axes[0].set_title(f'Аппарат #{aircraft_id} — признаки и аномалии (красное)')
    axes[-1].set_xlabel('Время (индекс)')
    plt.tight_layout()
    plt.show()

    # --- ГРАФИК 2: ошибка реконструкции ---
    plt.figure(figsize=(15, 4))
    err_series = pd.Series(aircraft_errors, index=aircraft_idx)
    plt.plot(err_series.index, err_series.values,
             color='steelblue', label='ошибка')
    plt.axhline(threshold, color='red', linestyle='--', label='порог')
    plt.scatter(anom_idx, aircraft_errors[anom_mask],
                color='red', s=20, zorder=5, label='аномалия')
    plt.title(f'Аппарат #{aircraft_id} — ошибка реконструкции')
    plt.xlabel('Время (индекс)'); plt.ylabel('MSE')
    plt.legend(); plt.grid(alpha=0.3)
    plt.show()

    # --- ГРАФИК 3: какие признаки виноваты (барчарт) ---
    sub_seq_mask = mask & np.isin(all_idx, anom_idx)
    anom_windows = X_all_seq[np.isin(all_idx, anom_idx) & mask]
    if len(anom_windows) > 0:
        model.eval()
        with torch.no_grad():
            sample = torch.tensor(anom_windows, dtype=torch.float32).to(device)
            recon = model(sample)
            feat_error = torch.mean((recon - sample)**2, dim=1).cpu().numpy()
        mean_feat_error = feat_error.mean(axis=0)

        plt.figure(figsize=(10, 6))
        order = np.argsort(mean_feat_error)
        plt.barh(np.array(features)[order],
                 mean_feat_error[order], color='indianred')
        plt.title(f'Аппарат #{aircraft_id} — какие признаки вызывают аномалии')
        plt.xlabel('Средняя ошибка по признаку')
        plt.tight_layout()
        plt.show()

        # текстом — топ-3 виновника
        top3 = np.array(features)[np.argsort(mean_feat_error)[-3:]][::-1]
        print(f"\n>>> Главные причины аномалий: {', '.join(top3)}")

    return {'id': aircraft_id, 'n_anomalies': n_anom,
            'anom_idx': anom_idx, 'threshold': threshold}

In [ ]:
# Анализ конкретного аппарата
result = analyze_aircraft(
    df_all, all_errors, all_idx, all_ids,
    aircraft_id=5,          # <<< id нужного аппарата
    features=features,
    percentile=PERCENTILE
)

In [ ]:
def summary_all_aircraft(df_all, all_errors, all_idx, all_ids,
                         percentile=99):
    """Сводная таблица по всем аппаратам."""
    rows = []
    for aid in sorted(df_all[ID_COLUMN].unique()):
        mask = all_ids == aid
        errs = all_errors[mask]
        if len(errs) == 0:
            continue
        threshold = np.percentile(errs, percentile)
        n_anom = (errs > threshold).sum()
        rows.append({
            'aircraft_id': aid,
            'окон': len(errs),
            'аномалий': n_anom,
            'процент': round(100 * n_anom / len(errs), 2),
            'макс_ошибка': round(errs.max(), 4),
            'средняя_ошибка': round(errs.mean(), 4),
        })
    summary = pd.DataFrame(rows).sort_values('макс_ошибка', ascending=False)
    return summary

summary = summary_all_aircraft(df_all, all_errors, all_idx, all_ids, PERCENTILE)
print(summary.to_string(index=False))

# Визуализация: какие аппараты самые проблемные
plt.figure(figsize=(10, 12))
plt.barh(summary['aircraft_id'].astype(str),
         summary['макс_ошибка'], color='indianred')
plt.title('Аппараты по максимальной ошибке (сверху — самые проблемные)')
plt.xlabel('Максимальная ошибка реконструкции')
plt.ylabel('ID аппарата')
plt.tight_layout()
plt.show()

In [ ]:
# Анализ ВСЕХ аппаратов по очереди (с графиками)
for aid in sorted(df_all[ID_COLUMN].unique()):
    analyze_aircraft(df_all, all_errors, all_idx, all_ids,
                     aircraft_id=aid, features=features,
                     percentile=PERCENTILE, show_plots=True)

In [ ]:
Или только топ-5 проблемных (по сводке):
top_problem = summary.head(5)['aircraft_id'].values
for aid in top_problem:
    analyze_aircraft(df_all, all_errors, all_idx, all_ids,
                     aircraft_id=aid, features=features,
                     percentile=PERCENTILE)

In [ ]:
Шаг 1 — Сохраняем всё нужное (один раз после обучения)


import joblib

# 1. Веса модели
torch.save(model.state_dict(), 'lstm_autoencoder.pth')

# 2. Scaler — БЕЗ него новые данные не нормализовать правильно
joblib.dump(scaler, 'scaler.pkl')

# 3. Конфиг — параметры, чтобы воссоздать модель и нарезку
config = {
    'features': features,
    'seq_len': SEQ_LEN,
    'n_features': len(features),
    'embedding_dim': 32,
    'percentile': PERCENTILE,
    'id_column': ID_COLUMN,
    'time_column': TIME_COLUMN,
    'diff_features': ['высота', 'скорость', 'тангаж', 'крен'],
}
joblib.dump(config, 'config.pkl')

print("✅ Сохранено: lstm_autoencoder.pth, scaler.pkl, config.pkl")

In [ ]:
Шаг 2 — Загружаем и используем (БЕЗ обучения)


import torch, joblib
import numpy as np
import pandas as pd

# === ЗАГРУЗКА (модель уже обучена, просто грузим) ===
config = joblib.load('config.pkl')
scaler = joblib.load('scaler.pkl')

# класс модели должен быть определён в коде (скопируй class LSTMAutoencoder)
model = LSTMAutoencoder(
    n_features=config['n_features'],
    embedding_dim=config['embedding_dim'],
    seq_len=config['seq_len']
).to(device)
model.load_state_dict(torch.load('lstm_autoencoder.pth', map_location=device))
model.eval()   # режим предсказания

print("✅ Модель загружена, готова к работе (обучение НЕ требуется)")

In [ ]:
Шаг 3 — Готовая функция: применить к любым данным
def predict_anomalies(df, model, scaler, config):
    """
    Принимает сырой датафрейм, возвращает df с колонками
    'recon_error' и 'anomaly'. Обучение НЕ нужно.
    """
    id_col = config['id_column']
    time_col = config['time_column']
    features = config['features']
    seq_len = config['seq_len']
    percentile = config['percentile']

    df = df.sort_values([id_col, time_col]).reset_index(drop=True)

    # производные (как при обучении)
    for col in config['diff_features']:
        df[f'{col}_diff'] = df.groupby(id_col)[col].diff()
    df = df.fillna(0)

    # нормализация ТЕМ ЖЕ scaler (transform, не fit!)
    df[features] = scaler.transform(df[features])

    # нарезка окон
    X_seq, idx, ids = create_sequences_grouped(df, features, seq_len, id_col)

    # ошибка реконструкции
    errors = compute_errors(model, X_seq)

    # аномалии по каждому аппарату
    df['recon_error'] = np.nan
    df.loc[idx, 'recon_error'] = errors
    df['anomaly'] = False
    for aid in df[id_col].unique():
        m = ids == aid
        thr = np.percentile(errors[m], percentile)
        anom = idx[m][errors[m] > thr]
        df.loc[anom, 'anomaly'] = True

    return df, errors, idx, ids


# === ПРИМЕНЕНИЕ К НОВЫМ ДАННЫМ ===
new_df = pd.read_csv('new_data.csv')
result_df, errors, idx, ids = predict_anomalies(new_df, model, scaler, config)

print(f"Аномалий найдено: {result_df['anomaly'].sum()}")

# и сразу анализ по конкретному аппарату:
analyze_aircraft(result_df, errors, idx, ids,
                 aircraft_id=3, features=config['features'],
                 percentile=config['percentile'])

In [ ]:
anomaly_detection/
├── model.py          # класс модели + общие функции (используется везде)
├── train.py          # обучение и сохранение модели
├── predict.py        # загрузка модели и поиск аномалий
├── visualize.py      # функции визуализации по аппаратам
├── config.pkl        # (создаётся после train.py)
├── scaler.pkl        # (создаётся после train.py)
└── lstm_autoencoder.pth  # (создаётся после train.py)

In [ ]:
Файл 1: model.py

Здесь модель и общие функции, которые нужны и при обучении, и при предсказании.

# model.py
import numpy as np
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class LSTMAutoencoder(nn.Module):
    """LSTM-автоэнкодер для детекции аномалий в многомерных временных рядах."""
    def __init__(self, n_features, embedding_dim=32, seq_len=30):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(n_features, embedding_dim, batch_first=True)
        self.decoder = nn.LSTM(embedding_dim, embedding_dim, batch_first=True)
        self.output_layer = nn.Linear(embedding_dim, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        latent = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(latent)
        return self.output_layer(decoded)


def create_sequences_grouped(df, features, seq_len, id_column, step=1):
    """Нарезает окна ОТДЕЛЬНО для каждого аппарата (не пересекая границы)."""
    all_seq, all_idx, all_ids = [], [], []
    for aid, group in df.groupby(id_column):
        group = group.sort_index()
        data = group[features].values
        gidx = group.index.values
        for i in range(0, len(data) - seq_len + 1, step):
            all_seq.append(data[i:i + seq_len])
            all_idx.append(gidx[i + seq_len - 1])
            all_ids.append(aid)
    return np.array(all_seq), np.array(all_idx), np.array(all_ids)


def compute_errors(model, X_seq, batch=256):
    """Считает ошибку реконструкции (MSE) для каждого окна."""
    model.eval()
    X_t = torch.tensor(X_seq, dtype=torch.float32)
    errors = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch):
            b = X_t[i:i+batch].to(device)
            recon = model(b)
            err = torch.mean((recon - b)**2, dim=(1, 2))
            errors.extend(err.cpu().numpy())
    return np.array(errors)


def add_diff_features(df, id_column, diff_features):
    """Добавляет производные (скорость изменения) признаков по каждому аппарату."""
    for col in diff_features:
        df[f'{col}_diff'] = df.groupby(id_column)[col].diff()
    return df.fillna(0)

In [ ]:
Файл 2: train.py

Обучает модель и сохраняет всё нужное. 

# train.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from model import (LSTMAutoencoder, create_sequences_grouped,
                   compute_errors, add_diff_features, device)

# ============ НАСТРОЙКИ (поменяй под свои данные) ============
DATA_PATH      = 'your_data.csv'
ID_COLUMN      = 'aircraft_id'
TIME_COLUMN    = 'time'
BASE_FEATURES  = [']  # + остальные 16
DIFF_FEATURES  = [']  # для производных
SEQ_LEN        = 30
EMBEDDING_DIM  = 32
EPOCHS         = 20
BATCH_SIZE     = 64
LR             = 1e-3
PERCENTILE     = 99
TEST_RATIO     = 0.2     # доля аппаратов в тест
# ============================================================

np.random.seed(42)
torch.manual_seed(42)


def main():
    # --- 1. Загрузка ---
    df = pd.read_csv(DATA_PATH)
    df = df.sort_values([ID_COLUMN, TIME_COLUMN]).reset_index(drop=True)
    print(f"Строк: {len(df)} | Аппаратов: {df[ID_COLUMN].nunique()}")

    # --- 2. Производные признаки ---
    df = add_diff_features(df, ID_COLUMN, DIFF_FEATURES)
    features = BASE_FEATURES + [f'{c}_diff' for c in DIFF_FEATURES]

    # --- 3. Деление train/test ПО АППАРАТАМ ---
    aircraft = df[ID_COLUMN].unique()
    np.random.shuffle(aircraft)
    split = int(len(aircraft) * (1 - TEST_RATIO))
    train_ac, test_ac = aircraft[:split], aircraft[split:]
    print(f"Train аппаратов: {len(train_ac)} | Test: {len(test_ac)}")

    df_train = df[df[ID_COLUMN].isin(train_ac)].copy()
    df_test  = df[df[ID_COLUMN].isin(test_ac)].copy()

    # --- 4. Нормализация (fit только на train!) ---
    scaler = StandardScaler()
    df_train[features] = scaler.fit_transform(df_train[features])
    df_test[features]  = scaler.transform(df_test[features])

    # --- 5. Нарезка окон ---
    X_train, _, _ = create_sequences_grouped(df_train, features, SEQ_LEN, ID_COLUMN)
    X_test, _, _  = create_sequences_grouped(df_test, features, SEQ_LEN, ID_COLUMN)
    print(f"Окон train: {X_train.shape} | test: {X_test.shape}")

    # --- 6. Модель ---
    model = LSTMAutoencoder(len(features), EMBEDDING_DIM, SEQ_LEN).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32)),
                              batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32)),
                             batch_size=BATCH_SIZE, shuffle=False)

    # --- 7. Обучение с валидацией + early stopping ---
    best_val = np.inf
    patience, no_improve = 5, 0
    hist_tr, hist_val = [], []

    for epoch in range(EPOCHS):
        model.train()
        tr_loss = 0
        for (batch,) in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch), batch)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for (batch,) in test_loader:
                batch = batch.to(device)
                val_loss += criterion(model(batch), batch).item()

        tr_loss /= len(train_loader)
        val_loss /= len(test_loader)
        hist_tr.append(tr_loss); hist_val.append(val_loss)
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | train {tr_loss:.5f} | val {val_loss:.5f}")

        # early stopping
        if val_loss < best_val:
            best_val = val_loss
            no_improve = 0
            torch.save(model.state_dict(), 'lstm_autoencoder.pth')  # лучшая модель
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping — модель перестала улучшаться")
                break

    # --- 8. График обучения ---
    plt.figure(figsize=(8, 3))
    plt.plot(hist_tr, label='train')
    plt.plot(hist_val, label='val')
    plt.title('Loss при обучении'); plt.xlabel('Эпоха'); plt.ylabel('MSE')
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig('training_loss.png', dpi=100, bbox_inches='tight')
    plt.show()

    # --- 9. Сохраняем scaler и config ---
    joblib.dump(scaler, 'scaler.pkl')
    config = {
        'features': features,
        'base_features': BASE_FEATURES,
        'diff_features': DIFF_FEATURES,
        'seq_len': SEQ_LEN,
        'n_features': len(features),
        'embedding_dim': EMBEDDING_DIM,
        'percentile': PERCENTILE,
        'id_column': ID_COLUMN,
        'time_column': TIME_COLUMN,
    }
    joblib.dump(config, 'config.pkl')

    print("\n✅ Готово! Сохранено:")
    print("   - lstm_autoencoder.pth (веса модели)")
    print("   - scaler.pkl (нормализация)")
    print("   - config.pkl (параметры)")


if __name__ == '__main__':
    main()

In [ ]:
Файл 3: predict.py

Загружает модель и ищет аномалии.

In [ ]:
# predict.py
import numpy as np
import pandas as pd
import joblib
import torch

from model import (LSTMAutoencoder, create_sequences_grouped,
                   compute_errors, add_diff_features, device)


def load_model():
    """Загружает обученную модель, scaler и config."""
    config = joblib.load('config.pkl')
    scaler = joblib.load('scaler.pkl')
    model = LSTMAutoencoder(
        n_features=config['n_features'],
        embedding_dim=config['embedding_dim'],
        seq_len=config['seq_len']
    ).to(device)
    model.load_state_dict(torch.load('lstm_autoencoder.pth', map_location=device))
    model.eval()
    print("✅ Модель загружена (обучение не требуется)")
    return model, scaler, config


def predict_anomalies(df, model, scaler, config):
    """
    Принимает сырой df, возвращает:
      df с колонками 'recon_error' и 'anomaly',
      errors, idx, ids — для последующего анализа.
    Аномалии считаются по КАЖДОМУ аппарату отдельно.
    """
    id_col   = config['id_column']
    time_col = config['time_column']
    features = config['features']
    seq_len  = config['seq_len']
    pct      = config['percentile']

    df = df.sort_values([id_col, time_col]).reset_index(drop=True)
    df = add_diff_features(df, id_col, config['diff_features'])

    # нормализация ТЕМ ЖЕ scaler (transform, не fit!)
    df[features] = scaler.transform(df[features])

    # нарезка окон + ошибка
    X_seq, idx, ids = create_sequences_grouped(df, features, seq_len, id_col)
    errors = compute_errors(model, X_seq)

    # аномалии по каждому аппарату
    df['recon_error'] = np.nan
    df.loc[idx, 'recon_error'] = errors
    df['anomaly'] = False
    for aid in df[id_col].unique():
        m = ids == aid
        if m.sum() == 0:
            continue
        thr = np.percentile(errors[m], pct)
        anom = idx[m][errors[m] > thr]
        df.loc[anom, 'anomaly'] = True

    return df, errors, idx, ids, X_seq


if __name__ == '__main__':
    # === применение к данным ===
    model, scaler, config = load_model()

    df = pd.read_csv('your_data.csv')   # любой датасет (старый или новый)
    df, errors, idx, ids, X_seq = predict_anomalies(df, model, scaler, config)

    print(f"\nВсего аномалий: {df['anomaly'].sum()}")

    # сводка по аппаратам
    from visualize import summary_all_aircraft
    summary = summary_all_aircraft(df, errors, ids, config)
    print("\nСводка по аппаратам:")
    print(summary.to_string(index=False))

    # сохраняем результат
    df.to_csv('result_with_anomalies.csv', index=False)
    print("\n✅ Результат сохранён в result_with_anomalies.csv")

In [ ]:
Файл 4: visualize.py

Функции для анализа и визуализации по каждому аппарату.

# visualize.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import device


def summary_all_aircraft(df, errors, ids, config):
    """Сводная таблица: аномалии по каждому аппарату."""
    id_col = config['id_column']
    pct = config['percentile']
    rows = []
    for aid in sorted(df[id_col].unique()):
        m = ids == aid
        errs = errors[m]
        if len(errs) == 0:
            continue
        thr = np.percentile(errs, pct)
        n_anom = int((errs > thr).sum())
        rows.append({
            'aircraft_id': aid,
            'окон': len(errs),
            'аномалий': n_anom,
            'процент': round(100 * n_anom / len(errs), 2),
            'макс_ошибка': round(float(errs.max()), 4),
            'средняя_ошибка': round(float(errs.mean()), 4),
        })
    return pd.DataFrame(rows).sort_values('макс_ошибка', ascending=False)


def analyze_aircraft(df, errors, idx, ids, X_seq, model, config,
                     aircraft_id, show_plots=True):
    """Полный анализ аномалий для ОДНОГО аппарата: текст + графики."""
    id_col   = config['id_column']
    features = config['features']
    base_feat = config['base_features']
    pct      = config['percentile']

    print("=" * 60)
    print(f"  АППАРАТ #{aircraft_id}")
    print("=" * 60)

    sub = df[df[id_col] == aircraft_id].copy()
    mask = ids == aircraft_id
    a_errors = errors[mask]
    a_idx = idx[mask]

    if len(a_errors) == 0:
        print("Нет данных для этого аппарата")
        return None

    threshold = np.percentile(a_errors, pct)
    anom_mask = a_errors > threshold
    anom_idx = a_idx[anom_mask]
    n_anom = int(anom_mask.sum())

    print(f"Всего окон:       {len(a_errors)}")
    print(f"Аномальных окон:  {n_anom} ({100*n_anom/len(a_errors):.1f}%)")
    print(f"Порог ошибки:     {threshold:.4f}")
    print(f"Макс. ошибка:     {a_errors.max():.4f}")
    print(f"Средняя ошибка:   {a_errors.mean():.4f}")

    if not show_plots or n_anom == 0:
        if n_anom == 0:
            print("Аномалий не обнаружено.")
        return {'id': aircraft_id, 'n_anomalies': n_anom, 'anom_idx': anom_idx}

    anom_rows = df.loc[anom_idx]

    # ГРАФИК 1: признаки с аномалиями
    fig, axes = plt.subplots(len(base_feat), 1, figsize=(15, 10), sharex=True)
    for ax, col in zip(axes, base_feat):
        ax.plot(sub.index, sub[col], color='steelblue', lw=0.8)
        ax.scatter(anom_rows.index, anom_rows[col], color='red', s=20, zorder=5)
        ax.set_ylabel(col); ax.grid(alpha=0.3)
    axes[0].set_title(f'Аппарат #{aircraft_id} — признаки и аномалии (красное)')
    axes[-1].set_xlabel('Время (индекс)')
    plt.tight_layout(); plt.show()

    # ГРАФИК 2: ошибка реконструкции
    plt.figure(figsize=(15, 4))
    plt.plot(a_idx, a_errors, color='steelblue', label='ошибка')
    plt.axhline(threshold, color='red', linestyle='--', label='порог')
    plt.scatter(anom_idx, a_errors[anom_mask], color='red', s=20, zorder=5,
                label='аномалия')
    plt.title(f'Аппарат #{aircraft_id} — ошибка реконструкции')
    plt.xlabel('Время (индекс)'); plt.ylabel('MSE')
    plt.legend(); plt.grid(alpha=0.3); plt.show()

    # ГРАФИК 3: какие признаки виноваты
    sel = mask & np.isin(idx, anom_idx)
    anom_windows = X_seq[sel]
    if len(anom_windows) > 0:
        model.eval()
        with torch.no_grad():
            sample = torch.tensor(anom_windows, dtype=torch.float32).to(device)
            recon = model(sample)
            feat_error = torch.mean((recon - sample)**2, dim=1).cpu().numpy()
        mean_fe = feat_error.mean(axis=0)
        order = np.argsort(mean_fe)
        plt.figure(figsize=(10, 6))
        plt.barh(np.array(features)[order], mean_fe[order], color='indianred')
        plt.title(f'Аппарат #{aircraft_id} — какие признаки вызывают аномалии')
        plt.xlabel('Средняя ошибка по признаку')
        plt.tight_layout(); plt.show()

        top3 = np.array(features)[np.argsort(mean_fe)[-3:]][::-1]
        print(f"\n>>> Главные причины аномалий: {', '.join(top3)}")

    return {'id': aircraft_id, 'n_anomalies': n_anom, 'anom_idx': anom_idx}

In [ ]:

Шаг 3 — анализ конкретного аппарата
from predict import load_model, predict_anomalies
from visualize import analyze_aircraft, summary_all_aircraft
import pandas as pd

# загрузка (без обучения!)
model, scaler, config = load_model()

# данные
df = pd.read_csv('your_data.csv')
df, errors, idx, ids, X_seq = predict_anomalies(df, model, scaler, config)

# сводка по всем
summary = summary_all_aircraft(df, errors, ids, config)
print(summary)

# детальный анализ одного аппарата
analyze_aircraft(df, errors, idx, ids, X_seq, model, config,
                 aircraft_id=17)   # <<< id нужного аппарата